# Laboratorio 2 - Complejidad y búsqueda de hiperparámetros

## AlpesPlanck

**Integrantes:**
- Daniel Esteban Pardo Pardo
- Samuel Andrés Molina Luna

**Curso:** ISIS2611 - Aprendizaje de Máquina
**Fecha:** [Fecha de entrega]

# 1. Contexto

En el Laboratorio 1 se construyó para AlpesPlanck un primer modelo de regresión lineal para
estimar la temperatura máxima del día siguiente en la estación de Jena, Alemania, identificando
las variables más influyentes y reflexionando sobre posibles fuentes de sesgo.

En este laboratorio se busca fortalecer ese primer acercamiento evaluando otros enfoques de
modelado (regresión polinomial y regularizada), analizando el efecto de la complejidad sobre la
capacidad de generalización, y cuantificando la incertidumbre del modelo mediante intervalos de
confianza obtenidos por remuestreo (bootstrapping).

Se reutiliza el mismo conjunto de datos del Laboratorio 1, aplicando el mismo proceso de
limpieza y preparación ya validado (Modelo 2 / `df_m2`), de forma que el énfasis de este
laboratorio esté en el modelado, la validación y el análisis del desempeño predictivo.

# 2. Carga de datos y librerías

### Librerías a usar en el transcurso del laboratorio
Librerías de Python para el procesamiento, modelado y análisis de datos:

- Pandas
- Scikit-Learn
- Matplotlib, Seaborn
- NumPy

In [ ]:
# Importación de las librerías a usar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, validation_curve, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer

from sklearn import set_config
set_config(display="diagram")

np.random.seed(42)

### Cargar datos a DataFrames

In [ ]:
# Dataset de entrenamiento (mismo del Laboratorio 1)
data = pd.read_csv('data/Datos Lab 1.csv')
data.head()

In [ ]:
# Dataset de prueba / producción (sin etiqueta), igual que en el Laboratorio 1
data_test = pd.read_csv('data/Datos Test Lab 1.csv')
data_test.head()

# 3. Reconstrucción del conjunto de datos limpio (Modelo 2 del Laboratorio 1)

Tal como indica el enunciado, este laboratorio no repite la exploración ni el procesamiento de
datos ya realizado en el Laboratorio 1 — se reutiliza directamente la versión resultante de esa
limpieza y preparación (el `df_m2` del Laboratorio 1, correspondiente al modelo optimizado).

A continuación se reproducen, de forma resumida y sin volver a justificar cada decisión, los
mismos pasos de limpieza aplicados en el Laboratorio 1, para partir exactamente del mismo punto.

**3.1 Unicidad — eliminación de duplicados**

In [ ]:
df_base = data.copy()
len_inicial = len(df_base)

# Eliminar duplicados exactos
df_base = df_base.drop_duplicates()
duplicados_exactos = len_inicial - len(df_base)

# Manejar fechas duplicadas: conservar la que tenga menos valores NaN
df_base['num_nans'] = df_base.isnull().sum(axis=1)
df_base = df_base.sort_values(by=['fecha', 'num_nans'])
df_base = df_base.drop_duplicates(subset=['fecha'], keep='first')
df_base = df_base.drop(columns=['num_nans'])

print(f"Registros tras depuración de duplicados: {len(df_base)}")

**3.2 Completitud — eliminación de registros sin variable objetivo**

In [ ]:
df_base = df_base.dropna(subset=['temp_max_manana'])
print(f"Registros con temp_max_manana no nula: {len(df_base)}")

**3.3 Validez — tratamiento de valores anómalos y centinela**

In [ ]:
# Presión atmosférica (Dominio real ~940 a 1040 mbar)
cols_presion = [c for c in df_base.columns if 'presion' in c and 'desv' not in c and 'std' not in c]
for col in cols_presion:
    df_base.loc[(df_base[col] < 940) | (df_base[col] > 1040), col] = np.nan

# Humedad relativa (Clipping al 100% y depuración de 0s artificiales)
cols_humedad = [c for c in df_base.columns if 'humedad' in c]
for col in cols_humedad:
    df_base.loc[df_base[col] > 100, col] = 100.0
    df_base.loc[df_base[col] <= 0.05, col] = np.nan

# Viento, ráfagas y desviaciones estándar
cols_viento_rafaga = [c for c in df_base.columns if 'viento' in c or 'rafaga' in c]
for col in cols_viento_rafaga:
    if 'desv' in col:
        df_base.loc[df_base[col] < 0, col] = np.nan
    if 'media' in col or 'min' in col or 'max' in col:
        df_base.loc[df_base[col] < 0, col] = np.nan

df_base.loc[df_base['viento_desv'] > 100, 'viento_desv'] = np.nan
df_base.loc[df_base['viento_norte'] < -100, 'viento_norte'] = np.nan

# Registros del día (máximo 144 mediciones por día)
if 'registros_del_dia' in df_base.columns:
    df_base.loc[df_base['registros_del_dia'] > 144, 'registros_del_dia'] = 144

# Depuración de la variable objetivo
df_base.loc[df_base['temp_max_manana'] > 50.0, 'temp_max_manana'] = np.nan
df_base = df_base.dropna(subset=['temp_max_manana'])

print(f"Registros finales tras depuración de valores anómalos: {len(df_base)}")

**3.4 Consistencia — normalización de variables categóricas**

In [ ]:
# Eliminar la columna 'fecha' por redundancia con dia, mes y anio
if 'fecha' in df_base.columns:
    df_base = df_base.drop(columns=['fecha'])

cat_cols_raw = df_base.select_dtypes(include=['object']).columns

for col in cat_cols_raw:
    mask_nulls = df_base[col].isnull()
    df_base[col] = df_base[col].astype(str).str.lower().str.strip()

    if col == 'mes':
        mapeo_meses = {
            'january': 'enero', 'february': 'febrero', 'march': 'marzo',
            'april': 'abril', 'may': 'mayo', 'june': 'junio',
            'july': 'julio', 'august': 'agosto', 'september': 'septiembre',
            'october': 'octubre', 'november': 'noviembre', 'december': 'diciembre'
        }
        df_base[col] = df_base[col].replace(mapeo_meses)

    if col == 'estacion_anio':
        mapeo_estaciones = {
            'summer': 'verano', 'verano': 'verano', 'berano': 'verano', 'verno': 'verano',
            'winter': 'invierno', 'invierno': 'invierno', 'invernio': 'invierno',
            'spring': 'primavera', 'primavera': 'primavera', 'primaveraa': 'primavera', 'primav': 'primavera',
            'autumn': 'otoño', 'fall': 'otoño', 'otoño': 'otoño', 'otono': 'otoño'
        }
        df_base[col] = df_base[col].replace(mapeo_estaciones)
        valores_invalidos = ['inverano', 'estacion_desconocida', 'verano_invierno', 'east']
        df_base[col] = df_base[col].replace(valores_invalidos, np.nan)

    if col == 'sector_viento':
        mapeo_viento = {
            'southwest': 'so', 'suroeste': 'so', 'sw': 'so',
            'northwest': 'no', 'noroeste': 'no', 'nw': 'no',
            'southeast': 'se', 'sudeste': 'se', 'sureste': 'se',
            'northeast': 'ne', 'noreste': 'ne',
            'south': 's', 'sur': 's',
            'north': 'n', 'norte': 'n',
            'east': 'e', 'este': 'e',
            'west': 'o', 'oeste': 'o'
        }
        df_base[col] = df_base[col].replace(mapeo_viento)

    df_base.loc[mask_nulls | df_base[col].isin(['nan', 'none', '']), col] = np.nan

print("Normalización de variables categóricas completa.")

**3.5 Ingeniería de características cíclicas y eliminación de colinealidad (Modelo 2 / `df_m2`)**

Se reproduce la versión optimizada del Laboratorio 1: transformación cíclica de `dia_del_anio`
y `direccion_viento`, y eliminación de las 5 variables redundantes detectadas por alta
correlación (r ≥ 0.80).

In [ ]:
# Copia de seguridad para trabajar sobre el dataset final del Laboratorio 1
df_m2 = df_base.copy()

# 1. Transformación cíclica para el día del año (Período: 365.25)
col_dia = 'dia_del_anio'
df_m2['dia_sin'] = np.sin(2 * np.pi * df_m2[col_dia] / 365.25)
df_m2['dia_cos'] = np.cos(2 * np.pi * df_m2[col_dia] / 365.25)
df_m2 = df_m2.drop(columns=[col_dia])

# 2. Transformación cíclica para la dirección del viento (Período: 360°)
col_viento = 'direccion_viento'
df_m2['direccion_viento_sin'] = np.sin(2 * np.pi * df_m2[col_viento] / 360.0)
df_m2['direccion_viento_cos'] = np.cos(2 * np.pi * df_m2[col_viento] / 360.0)
df_m2 = df_m2.drop(columns=[col_viento])

# 3. Eliminación de variables redundantes (r >= 0.80 con otras variables)
cols_a_eliminar = ['presion_min', 'presion_max', 'humedad_min', 'viento_max', 'rafaga_desv']
df_m2 = df_m2.drop(columns=cols_a_eliminar)

print(f"df_m2 final: {df_m2.shape[0]} filas, {df_m2.shape[1]} columnas")
df_m2.head()

**3.6 Separación de variables y partición train/test**

In [ ]:
# Separar características (X) y variable objetivo (y)
X = df_m2.drop(columns=['temp_max_manana'])
y = df_m2['temp_max_manana']

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f'Columnas numéricas ({len(num_cols)}): {num_cols}')
print(f'Columnas categóricas ({len(cat_cols)}): {cat_cols}')

In [ ]:
# Partición train/test (misma configuración usada en el Laboratorio 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")